In [ ]:
import json
import time
import re
from datetime import datetime, timedelta
from urllib.parse import urljoin
from typing import List, Dict, Any, Optional, Set

import cloudscraper
from bs4 import BeautifulSoup

In [ ]:
# =============================================================================
# КОНФИГУРАЦИЯ
# =============================================================================

SPRINGER_BASE = "https://link.springer.com"
SPRINGER_VOLUME_URL = f"{SPRINGER_BASE}/journal/13321/articles?filter-by-volume=18&sortBy=newestFirst"
TARGET_CHEMISTRY = 100

BIORXIV_API = "https://api.biorxiv.org"
TARGET_BIOLOGY = 100
DAYS_BACK = 30

# можно добавить и другие
BIORXIV_CATEGORIES = [
    "genomics",
    "neuroscience",
    "cell-biology",
    "biochemistry",
    "evolutionary-biology",
]

OUTPUT_FILE = "chem_bio_data.json"

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,ru;q=0.8",
}

In [ ]:
# =============================================================================
# ХИМИЯ
# =============================================================================

def get_springer_article_links(page_url: str) -> List[str]:
    links = []
    try:
        resp = scraper.get(page_url, headers=HEADERS, timeout=60)

        if resp.status_code != 200:
            return []

        soup = BeautifulSoup(resp.text, "html.parser")
        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"]
            if "/article/10.1186/" in href:
                full_url = urljoin(SPRINGER_BASE, href)
                if full_url not in links:
                    links.append(full_url)

        if not links:
            for a_tag in soup.select("h3 a[href*='/article/']"):
                full_url = urljoin(SPRINGER_BASE, a_tag["href"])
                if full_url not in links:
                    links.append(full_url)
    except Exception as e:
        print(f"Ошибка получения списка ссылок: {e}")
    return links

def parse_springer_article(article_url: str) -> Optional[Dict[str, Any]]:
    try:
        resp = scraper.get(article_url, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else ""
        if not title: return None

        authors = []
        for a_tag in soup.find_all("a", attrs={"data-test": "author-name"}):
            name = a_tag.get_text(strip=True)
            if name and name not in authors: authors.append(name)
        if not authors:
            for li in soup.select("ul.c-article-author-list li"):
                name = re.sub(r'^\d+,?\s*', '', li.get_text(strip=True)).strip()
                if name and name not in authors: authors.append(name)
        author_str = "; ".join(authors) if authors else "Unknown"

        full_text_parts = []
        article_body = soup.find("div", class_="c-article-body")
        if article_body:
            for p in article_body.find_all("p"):
                text = p.get_text(strip=True)
                if text and len(text) > 20: full_text_parts.append(text)
        else:
            for p in soup.find_all("p"):
                text = p.get_text(strip=True)
                if text and len(text) > 50: full_text_parts.append(text)

        full_text = "\n\n".join(full_text_parts)

        return {
            "category": "chemistry",
            "subcategory": "Journal of Cheminformatics",
            "title": title,
            "author": author_str,
            "text": full_text if full_text else "Full text not available.",
            "source_url": article_url,
        }
    except Exception as e:
        print(f"Ошибка парсинга {article_url}: {e}")
        return None

def fetch_chemistry_articles() -> List[Dict[str, Any]]:
    print(f"\n[Химия] Цель: {TARGET_CHEMISTRY} статей.")
    new_articles = []
    seen_urls: Set[str] = set()
    page = 1

    while len(new_articles) < TARGET_CHEMISTRY and page <= 15:
        separator = "&" if "?" in SPRINGER_VOLUME_URL else "?"
        page_url = f"{SPRINGER_VOLUME_URL}{separator}page={page}"

        print(f"  Страница {page}:")
        links = get_springer_article_links(page_url)
        if not links:
            print("  Ссылки не найдены. Остановка.")
            break

        for link in links:
            if len(new_articles) >= TARGET_CHEMISTRY: break

            # Защита от дублей внутри одного запуска
            if link in seen_urls:
                print(f"    [Пропуск] Дубль в этом запуске: {link}")
                continue

            print(f"    Парсинг: {link}")
            article = parse_springer_article(link)
            if article:
                new_articles.append(article)
                seen_urls.add(link)
                print(f"      Успешно (текст: {len(article['text'])} симв.)")
            time.sleep(2.5)

        page += 1
        time.sleep(4.0)

    return new_articles

In [ ]:
# =============================================================================
# БИОЛОГИЯ
# =============================================================================

def parse_biorxiv_fulltext(article_url: str) -> str:
    try:
        resp = scraper.get(article_url+'.full', headers=HEADERS, timeout=60)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        full_text_parts = []
        main_content = soup.find("div", class_="article-chunk")

        if main_content:
            for element in main_content.find_all(["p", "h2", "h3", "li"]):
                text = element.get_text(strip=True)
                if text and len(text) > 15:
                    full_text_parts.append(text)
        else:
            for p in soup.find_all("p"):
                text = p.get_text(strip=True)
                if text and len(text) > 50:
                    full_text_parts.append(text)

        return " ".join(full_text_parts)
    except Exception as e:
        print(f"      [!] Ошибка скачивания текста bioRxiv: {e}")
        return ""

def fetch_biorxiv_articles() -> List[Dict[str, Any]]:
    print(f"\n[Биология] Цель: {TARGET_BIOLOGY} статей.")

    # Вычисляем дату 30 дней назад
    end_date = datetime.now()
    start_date = end_date - timedelta(days=DAYS_BACK)
    date_from = start_date.strftime("%Y-%m-%d")
    date_to = end_date.strftime("%Y-%m-%d")

    print(f"  Ищем статьи, опубликованные с {date_from} по {date_to}")

    all_preprints = []
    cursor = 0

    # Загружаем метаданные
    while len(all_preprints) < (TARGET_BIOLOGY * 3) and cursor < 500:
        url = f"{BIORXIV_API}/details/biorxiv/{date_from}/{date_to}/{cursor}"
        try:
            resp = scraper.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  [bioRxiv API] Ошибка: {e}")
            break

        batch = data.get("collection", [])
        if not batch:
            break

        all_preprints.extend(batch)
        cursor += 100
        time.sleep(1.5)

    print(f"  Загружено метаданных всего: {len(all_preprints)}")

    # Фильтруем по категориям
    filtered = [p for p in all_preprints if any(cat.lower() in p.get("category", "").lower() for cat in BIORXIV_CATEGORIES)]

    valid_date_preprints = []
    for p in filtered:
        p_date_str = p.get("date", "")
        if p_date_str and p_date_str >= date_from:
            valid_date_preprints.append(p)

    print(f"  Статей в нужных категориях за последний месяц: {len(valid_date_preprints)}")

    # Сортируем по дате
    valid_date_preprints.sort(key=lambda x: x.get("date", ""), reverse=True)

    new_articles = []
    seen_dois: Set[str] = set()

    for paper in valid_date_preprints:
        if len(new_articles) >= TARGET_BIOLOGY:
            break

        doi = paper.get("doi", "")
        if not doi:
            continue

        # Защита от дублей внутри одного запуска
        if doi in seen_dois:
            print(f"    [Пропуск] Дубль в этом запуске: DOI {doi}")
            continue

        version = paper.get("version", 1)
        article_url = f"https://www.biorxiv.org/content/{doi}v{version}"

        print(f"    Парсинг: {doi} (Дата: {paper.get('date')})")
        full_text = parse_biorxiv_fulltext(article_url)

        text_content = full_text if full_text else (paper.get("abstract", "") or "No text available.")

        new_articles.append({
            "category": "biology",
            "subcategory": paper.get("category", "unknown"),
            "title": paper.get("title", ""),
            "author": paper.get("authors", ""),
            "text": text_content,
            "source_url": article_url,
        })

        seen_dois.add(doi)
        time.sleep(3.0)

    return new_articles

In [ ]:
# =============================================================================
# MAIN
# =============================================================================

def main():
    print("=" * 70)
    print("СБОР СТАТЕЙ: ХИМИЯ + БИОЛОГИЯ")
    print("=" * 70)

    # Сбор химии
    new_chem_articles = fetch_chemistry_articles()

    # Сбор биологии
    new_bio_articles = fetch_biorxiv_articles()

    all_articles = new_chem_articles + new_bio_articles

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_articles, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 70)
    print(f"  Всего записей в файле: {len(all_articles)}")
    print(f"    - Химия: {len(new_chem_articles)}")
    print(f"    - Биология: {len(new_bio_articles)}")
    print(f"  Файл сохранён: {OUTPUT_FILE}")
    print("=" * 70)

if __name__ == "__main__":
    main()